# Sentinel — QLoRA fine-tune of the alert triage classifier

Trains `Qwen2.5-1.5B-Instruct` to classify production alerts into a fixed
JSON schema, then exports it for Ollama.

**Runtime → Change runtime type → T4 GPU.** Roughly 25–35 minutes on a free T4.

## Why this notebook exists

This project was built on a machine with a 6GB RTX 3060, and the trainer in
`finetune/train_qlora.py` targets exactly that. It cannot run there: Windows
Smart App Control blocks the CUDA build of PyTorch (`shm.dll` and its
dependencies are unsigned), and disabling Smart App Control on Windows 11 is
irreversible without reinstalling the OS. The CPU wheel loads fine but would
turn a 30-minute run into an overnight one.

So the fine-tune runs here. The hyperparameters are identical to the local
script — a T4 has 16GB against the 3060's 6GB, so anything that fits locally
fits here with room to spare.

## 1. Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
%pip install -q -U "transformers>=4.48" "peft>=0.14" "trl>=0.13" \
    "datasets>=3.2" "accelerate>=1.2" "bitsandbytes>=0.45" sentencepiece

## 2. Get the dataset

Generated from templates rather than distilled from a frontier model. These
labels are definitional — a total outage is P1 because of what the alert says —
so template generation makes every label correct by construction and the whole
set reproducible from a seed.

Clone the repo to regenerate it, or upload `train_chat.jsonl` / `val_chat.jsonl`
from `finetune/data/` directly.

In [ ]:
import os

REPO = "https://github.com/GautamRaju18/Sentinel.git"

if not os.path.exists("Sentinel"):
    !git clone -q $REPO
%cd Sentinel
%pip install -q pydantic pydantic-settings
!python finetune/generate_dataset.py

In [ ]:
import json

def load(name):
    with open(f"finetune/data/{name}_chat.jsonl", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows, val_rows = load("train"), load("val")
print(f"train={len(train_rows)}  val={len(val_rows)}")
print(json.dumps(train_rows[0], indent=2)[:900])

## 3. Load the base model in 4-bit

NF4 with double quantisation. The base weights are frozen, so their precision
matters far less than the adapter's; double-quant saves roughly 0.4 bits per
parameter, which is what makes this fit in 6GB on the target hardware.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE, quantization_config=quant, device_map={"": 0},
    torch_dtype=torch.float16, trust_remote_code=True,
)
model.config.use_cache = False
print(f"loaded. GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4. Baseline: what does the model do before training?

Worth running. Without it you have no idea whether the fine-tune helped or
whether the base model could already do this.

In [ ]:
def classify(messages, max_new_tokens=140):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

sample = train_rows[0]["messages"]
print("--- before training ---")
print(classify(sample[:2]))
print("\n--- ground truth ---")
print(sample[2]["content"])

## 5. Attach the LoRA adapter

Rank 16 on attention **and** MLP projections. Attention-only adapters learn the
task but keep drifting on output format; the MLP projections are where the
"always emit this JSON shape" behaviour settles.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 6. Train

Loss on the **completion only**. Training on the prompt tokens too would spend
most of the gradient budget teaching the model to reproduce alert text it will
always be given.

In [ ]:
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_dataset(rows):
    return Dataset.from_dict({
        "text": [tokenizer.apply_chat_template(r["messages"], tokenize=False) for r in rows]
    })

config = SFTConfig(
    output_dir="outputs/triage-qwen1.5b-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    max_seq_length=1024,
    packing=False,
    dataset_text_field="text",
    report_to=[],
)

trainer = SFTTrainer(
    model=model, args=config,
    train_dataset=to_dataset(train_rows),
    eval_dataset=to_dataset(val_rows),
    processing_class=tokenizer,
)

try:
    from trl import DataCollatorForCompletionOnlyLM
    trainer.data_collator = DataCollatorForCompletionOnlyLM(
        response_template="<|im_start|>assistant\n", tokenizer=tokenizer
    )
    print("completion-only loss: enabled")
except Exception as e:
    print(f"completion-only loss unavailable ({e}); training on full sequence")

trainer.train()

## 7. After training

In [ ]:
print("--- after training ---")
print(classify(sample[:2]))
print("\n--- ground truth ---")
print(sample[2]["content"])

## 8. Score the held-out test set

The test split uses services that appear in **no** training example, so a model
that memorised service names scores badly here. That is the point of it.

`critical_underestimates` — a P1 filed as P3 or P4 — is the number that matters
operationally. An unattended P1 is an outage nobody is working on; the reverse
merely wastes attention.

In [ ]:
import json, re, time

with open("finetune/data/test.jsonl", encoding="utf-8") as f:
    test_rows = [json.loads(line) for line in f if line.strip()]

SYSTEM = train_rows[0]["messages"][0]["content"]
RANK = {"P1": 0, "P2": 1, "P3": 2, "P4": 3}

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

sev_ok = cat_ok = parsed = critical = 0
latencies = []

for row in test_rows:
    started = time.perf_counter()
    raw = classify([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Classify this alert:\n\n{row['alert']}"},
    ])
    latencies.append((time.perf_counter() - started) * 1000)
    pred, truth = extract_json(raw), row["label"]
    if not pred:
        continue
    parsed += 1
    if pred.get("severity") == truth["severity"]:
        sev_ok += 1
    if pred.get("category") == truth["category"]:
        cat_ok += 1
    if pred.get("severity") in RANK and RANK[pred["severity"]] - RANK[truth["severity"]] >= 2:
        critical += 1

n = len(test_rows)
latencies.sort()
print(f"n                        {n}")
print(f"valid JSON               {parsed/n*100:.1f}%")
print(f"severity accuracy        {sev_ok/n*100:.1f}%   (baseline 37.5%)")
print(f"category accuracy        {cat_ok/n*100:.1f}%   (baseline 25.0%)")
print(f"critical underestimates  {critical/n*100:.1f}%   (baseline 10.0%)")
print(f"latency p50              {latencies[len(latencies)//2]:.0f} ms")

## 9. Merge and export

Merge into **fp16**, not back into 4-bit — merging into quantised weights loses
most of what was learned.

In [ ]:
import gc
from peft import PeftModel

trainer.save_model("outputs/triage-qwen1.5b-lora")
tokenizer.save_pretrained("outputs/triage-qwen1.5b-lora")

del model, trainer
gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, "outputs/triage-qwen1.5b-lora").merge_and_unload()
merged.save_pretrained("merged/triage-qwen1.5b", safe_serialization=True)
tokenizer.save_pretrained("merged/triage-qwen1.5b")
print("merged")

In [ ]:
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp
%pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py merged/triage-qwen1.5b \
    --outfile sentinel-triage-f16.gguf --outtype f16
!ls -lh sentinel-triage-f16.gguf

## 10. Download and register locally

Download the GGUF, then on your own machine:

```bash
ollama create sentinel-triage -f Modelfile
```

with a `Modelfile` containing:

```
FROM ./sentinel-triage-f16.gguf
PARAMETER temperature 0
PARAMETER top_p 0.1
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
```

Then point the triage tier at it in `.env`:

```
MODEL_TRIAGE=ollama:sentinel-triage
TRIAGE_BACKEND=finetuned
```

and produce the comparison against the recorded baseline:

```bash
uv run python evals/triage_eval.py --out evals/results_finetuned.json
uv run python evals/compare.py
```

In [ ]:
from google.colab import files

files.download("sentinel-triage-f16.gguf")